# 0200_ingest_parquet_file
Lands one raw parquet file into Bronze exactly as delivered, plus three audit columns:
`_ingested_at`, `_source_file`, `_batch_id`.

- `load_type = SNAPSHOT`: the file is a full snapshot (reference data). The table holds the
  latest snapshot and `<table>_snapshots` keeps every delivery.
- `load_type = INCREMENTAL`: only rows above the Bronze table's highest `watermark_column` are
  appended (the ledger and the till feed). Against a live ERP the same watermark goes into the
  extract query; here the delivered file is filtered on read. Bronze is append-only, so every
  delivery stays on record with its `_batch_id`.

In [ ]:
# Bootstrap: find the pipeline root and load the shared notebooks.
# run_notebook() plays the part of Databricks' %run: it runs another notebook's code cells in
# this notebook's namespace. It is used instead of Jupyter's %run because that breaks on
# Windows paths with spaces.
import json
import pathlib

PIPELINE_ROOT = next(
    folder
    for folder in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (folder / "00_shared").is_dir()
)


def run_notebook(relative_path):
    notebook = json.loads((PIPELINE_ROOT / relative_path).read_text(encoding="utf-8"))
    for cell in notebook["cells"]:
        if cell["cell_type"] == "code":
            exec("".join(cell["source"]), globals())


if "SHARED_LOADED" not in globals():
    run_notebook("00_shared/000_nb_configurations.ipynb")
    run_notebook("00_shared/001_nb_utils.ipynb")
    SHARED_LOADED = True

## Loader

In [ ]:
def ingest_parquet_file(config, batch_id):
    source_folder = CONNECTIONS[config["source_connection_name"]]
    source_file = source_folder / config["source_object"]
    schema, table_name = config["destination_schema"], config["destination_table"]
    df = spark.read.parquet(source_file.as_posix())

    incremental = (
        config["load_type"] == "INCREMENTAL"
        and not config.get("full_refresh")
        and table_exists("bronze", schema, table_name)
    )
    if incremental:
        watermark = config["watermark_column"]
        last_watermark = max_value(table_full_name("bronze", schema, table_name), watermark)
        if last_watermark is not None:
            df = df.where(F.col(watermark) > F.lit(last_watermark))
        print(f"  {table_name}: {watermark} > {last_watermark}")

    df = (
        df
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.lit(source_file.name))
        .withColumn("_batch_id", F.lit(batch_id))
    )
    if config["load_type"] == "SNAPSHOT":
        return write_snapshot(df, "bronze", schema, table_name)
    if not incremental:
        return write_table(df, "bronze", schema, table_name)

    new_rows = df.count()
    if new_rows:
        df.write.format(TABLE_FORMAT).mode("append").save(table_path("bronze", schema, table_name).as_posix())
        reregister_table("bronze", schema, table_name)
    return new_rows


LOADERS["0200_ingest_parquet_file"] = ingest_parquet_file

## Run on its own

In [ ]:
# Parameters: set a control_id and run all to load one flow on its own (handy for testing).
# Left at 0, this notebook only defines its loader for the orchestrator.
control_id = 0
batch_id = ""

In [ ]:
if control_id:
    print(execute_flow(control_id, batch_id or new_batch_id(), ingest_parquet_file, triggered_by="manual"))